# CWS spatial-support audit and reference-domain sensitivity

Run this **after** the existing OWS reference, residual-risk, and QC × risk fusion notebooks. It does not retrain the OWS reference model or the residual-risk model. It audits whether the CWS results are stable across OWS-support domains and adds a conservative post-hoc guardrail for low/outside-support stations.

Outputs are written under `spatial_support_audit__<risk_model_key>`.


In [ ]:

from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Mapping, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


## 1. Configure paths

Set QC_DATA_ROOT, QC_PROJECT_ROOT, and QC_PROJECT_ID before running, or edit this cell. Use the **context_static_met** fusion manifest as the primary model.


In [ ]:
import os
PROJECT_ROOT = Path(os.environ.get("QC_PROJECT_ROOT", Path(os.environ.get("QC_DATA_ROOT", Path.cwd() / "data")) / os.environ.get("QC_PROJECT_ID", "project_id"))).expanduser().resolve()
BENCHMARK_DIR = PROJECT_ROOT / "results" / "20_quality_control" / "qc_benchmark"

RISK_MODEL_KEY = "context_static_met"
CITY = os.environ.get("QC_PROJECT_ID", "project_id")
REFERENCE_RUN_LABEL = "catboost_corrected_ows_metadata_iteration01"
RESIDUAL_RISK_RUN_LABEL = f"{REFERENCE_RUN_LABEL}__residual_risk_AUDITED__catboost__time_train"
FUSION_RUN_LABEL = f"{RESIDUAL_RISK_RUN_LABEL}__qc_risk_fusion_AUDITED__{RISK_MODEL_KEY}"

FUSION_MANIFEST_PATH = (
    BENCHMARK_DIR / "qc_risk_fusion_audited" / FUSION_RUN_LABEL / f"{CITY}_qc_risk_fusion_manifest.json"
)
OWS_REFERENCE_DIR = BENCHMARK_DIR / "ows_reference" / REFERENCE_RUN_LABEL
CWS_TO_OWS_PAIR_TABLE_PATH = OWS_REFERENCE_DIR / f"{CITY}_cws_to_ows_pair_table.csv"

OUTPUT_DIR = BENCHMARK_DIR / f"spatial_support_audit__{RISK_MODEL_KEY}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVALUATION_SPLIT = "test"

print("Fusion manifest:", FUSION_MANIFEST_PATH)
print("OWS reference dir:", OWS_REFERENCE_DIR)
print("Output dir:", OUTPUT_DIR)
assert FUSION_MANIFEST_PATH.exists(), f"Missing fusion manifest: {FUSION_MANIFEST_PATH}"


## 2. I/O helpers


In [ ]:

def load_json(path: str | Path) -> dict:
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_dataframe_auto(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")


def save_df(df: pd.DataFrame, name: str) -> Path:
    path = OUTPUT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".csv":
        df.to_csv(path, index=False)
    elif path.suffix.lower() == ".parquet":
        df.to_parquet(path, index=False)
    else:
        path = path.with_suffix(".csv")
        df.to_csv(path, index=False)
    print("saved:", path)
    return path


def first_existing(cols: Sequence[str], candidates: Sequence[str]) -> str | None:
    colset = set(map(str, cols))
    for c in candidates:
        if c in colset:
            return c
    return None


def as_bool(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(int).astype(bool)
    return s.astype("string").str.lower().isin(["true", "1", "yes", "y"])


def safe_num(df: pd.DataFrame, col: str, default=np.nan) -> pd.Series:
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(default, index=df.index, dtype="float64")


## 3. Load the fused CWS table

This loads the existing fused table, then filters to the test split. No model is retrained.


In [ ]:

fusion_manifest = load_json(FUSION_MANIFEST_PATH)
fused_path = Path(fusion_manifest["outputs"]["fused_path"])
print("Fused table:", fused_path)
assert fused_path.exists(), f"Missing fused parquet from manifest: {fused_path}"

fused_all = load_dataframe_auto(fused_path)
print("Loaded fused_all:", fused_all.shape)

if "split" in fused_all.columns:
    eval_df = fused_all[fused_all["split"].astype(str).eq(EVALUATION_SPLIT)].copy()
else:
    eval_df = fused_all.copy()

print("Evaluation rows:", eval_df.shape)
print("Columns:", len(eval_df.columns))


## 4. Add spatial OWS-support columns

This uses support columns already present in the fused table when available. If nearest-distance fields are absent, it merges the nearest OWS distance from `<CITY>_cws_to_ows_pair_table.csv`.


In [ ]:

def station_key_df(df: pd.DataFrame) -> pd.Series:
    return df["network"].astype(str).str.strip() + "::" + df["station_id"].astype(str).str.strip()


def nearest_distance_from_pair_table(pair_table_path: str | Path) -> pd.DataFrame:
    pair_table_path = Path(pair_table_path)
    if not pair_table_path.exists():
        raise FileNotFoundError(pair_table_path)

    usecols = [
        "target_station_id", "target_network", "source_station_id", "ref_rank", "ref_dist_km",
        "target_minus_source_elev_meters", "abs_target_minus_source_elev_meters",
        "target_minus_source_building_height_m", "abs_target_minus_source_building_height_m",
        "same_LC_point_lg", "same_LC_buffer_lg", "same_LCZ_point_lg", "same_LCZ_buffer_lg",
    ]
    header = pd.read_csv(pair_table_path, nrows=0).columns.tolist()
    usecols = [c for c in usecols if c in header]
    p = pd.read_csv(pair_table_path, usecols=usecols)

    p["target_station_id"] = p["target_station_id"].astype(str).str.strip()
    p["target_network"] = p["target_network"].astype(str).str.strip()
    p["ref_dist_km"] = pd.to_numeric(p["ref_dist_km"], errors="coerce")
    if "ref_rank" in p.columns:
        p["ref_rank"] = pd.to_numeric(p["ref_rank"], errors="coerce")
        p = p.sort_values(["target_network", "target_station_id", "ref_rank", "ref_dist_km"])
    else:
        p = p.sort_values(["target_network", "target_station_id", "ref_dist_km"])

    nearest = p.groupby(["target_network", "target_station_id"], as_index=False).first()
    nearest = nearest.rename(
        columns={
            "target_network": "network",
            "target_station_id": "station_id",
            "ref_dist_km": "nearest_ows_dist_km_from_pair_table",
            "source_station_id": "nearest_ows_station_id_from_pair_table",
        }
    )
    return nearest


def add_spatial_support_columns(df: pd.DataFrame, pair_table_path: str | Path | None = None) -> pd.DataFrame:
    x = df.copy()
    x["station_id"] = x["station_id"].astype(str).str.strip()
    x["network"] = x["network"].astype(str).str.strip()

    dist_col = first_existing(
        x.columns,
        [
            "nearest_ows_dist_km",
            "ref_support_nearest_dist_km",
            "ref_min_dist_km",
            "ref_nearest_dist_km",
            "ref_nearest_value_dist_km",
            "ows_nearest_dist_km",
            "ows_local_min_dist_km",
        ],
    )

    if dist_col is not None:
        x["nearest_ows_dist_km"] = pd.to_numeric(x[dist_col], errors="coerce")
        x["nearest_ows_dist_source"] = dist_col
    elif pair_table_path is not None and Path(pair_table_path).exists():
        nearest = nearest_distance_from_pair_table(pair_table_path)
        x = x.merge(nearest, on=["network", "station_id"], how="left", validate="many_to_one")
        x["nearest_ows_dist_km"] = pd.to_numeric(x["nearest_ows_dist_km_from_pair_table"], errors="coerce")
        x["nearest_ows_dist_source"] = f"{CITY}_cws_to_ows_pair_table.csv"
    else:
        x["nearest_ows_dist_km"] = np.nan
        x["nearest_ows_dist_source"] = "missing"


    if "ref_support_class" not in x.columns:
        dist = pd.to_numeric(x["nearest_ows_dist_km"], errors="coerce")
        n_local = safe_num(x, "ref_local_n")
        spread = safe_num(x, "ref_local_spread_c")

        support = pd.Series("moderate_support", index=x.index, dtype="string")
        support.loc[(dist <= 2.0) & (n_local >= 4) & ((spread <= 2.0) | spread.isna())] = "high_support"
        support.loc[(dist > 5.0) | (n_local <= 1) | (spread > 4.0)] = "low_support"
        support.loc[(dist > 10.0) | (n_local <= 0)] = "outside_support"
        support.loc[dist.isna() & n_local.isna() & spread.isna()] = "unknown_support"
        x["ref_support_class"] = support
    else:
        x["ref_support_class"] = x["ref_support_class"].astype("string").fillna("unknown_support")

    bins = [-np.inf, 1, 2, 5, 8, 10, np.inf]
    labels = ["<=1km", "1-2km", "2-5km", "5-8km", "8-10km", ">10km"]
    x["nearest_ows_distance_bin"] = pd.cut(
        pd.to_numeric(x["nearest_ows_dist_km"], errors="coerce"),
        bins=bins,
        labels=labels,
    ).astype("string").fillna("unknown")


    domain = pd.Series("unknown", index=x.index, dtype="string")
    d = pd.to_numeric(x["nearest_ows_dist_km"], errors="coerce")
    domain.loc[d <= 1] = "core_<=1km"
    domain.loc[(d > 1) & (d <= 5)] = "supported_1-5km"
    domain.loc[(d > 5) & (d <= 8)] = "moderate_5-8km"
    domain.loc[(d > 8) & (d <= 10)] = "low_8-10km"
    domain.loc[d > 10] = "outside_>10km"
    domain.loc[d.isna()] = "unknown"
    x["ows_distance_domain"] = domain

    domain_order = [
    "core_<=1km",
    "supported_1-5km",
    "moderate_5-8km",
    "low_8-10km",
    "outside_>10km",
    "unknown",
    ]

    x["ows_distance_domain"] = pd.Categorical(
        x["ows_distance_domain"],
        categories=domain_order,
        ordered=True,
    )


    support_class = x["ref_support_class"].astype(str)
    x["poor_ows_support_strict"] = (d > 10) | support_class.isin(["outside_support"])
    x["poor_ows_support_sensitive"] = (d > 8) | support_class.isin(["low_support", "outside_support"])

    return x


eval_df = add_spatial_support_columns(eval_df, pair_table_path=CWS_TO_OWS_PAIR_TABLE_PATH)
print(eval_df[["nearest_ows_dist_km", "nearest_ows_dist_source", "nearest_ows_distance_bin", "ows_distance_domain", "ref_support_class"]].head())
print(eval_df["nearest_ows_dist_source"].value_counts(dropna=False).head())


## 5. Basic support-domain counts


In [ ]:

def support_count_table(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    total_n = len(df)
    total_stations = df[["network", "station_id"]].drop_duplicates().shape[0]
    for key, g in df.groupby(group_col, dropna=False):
        rows.append({
            group_col: str(key),
            "n_rows": int(len(g)),
            "row_fraction": float(len(g) / total_n) if total_n else np.nan,
            "n_stations": int(g[["network", "station_id"]].drop_duplicates().shape[0]),
            "station_fraction": float(g[["network", "station_id"]].drop_duplicates().shape[0] / total_stations) if total_stations else np.nan,
            "median_nearest_ows_dist_km": float(pd.to_numeric(g["nearest_ows_dist_km"], errors="coerce").median()),
            "p95_nearest_ows_dist_km": float(pd.to_numeric(g["nearest_ows_dist_km"], errors="coerce").quantile(0.95)),
        })
    return pd.DataFrame(rows).sort_values("n_rows", ascending=False).reset_index(drop=True)

support_by_bin = support_count_table(eval_df, "nearest_ows_distance_bin")
support_by_domain = support_count_table(eval_df, "ows_distance_domain")
support_by_class = support_count_table(eval_df, "ref_support_class")

save_df(support_by_bin, f"{CITY}_spatial_support_counts_by_nearest_ows_distance_bin.csv")
save_df(support_by_domain, f"{CITY}_spatial_support_counts_by_distance_domain.csv")
save_df(support_by_class, f"{CITY}_spatial_support_counts_by_ref_support_class.csv")

display(support_by_domain)
display(support_by_class)


## 6. Policy and correction metrics by support domain

These tables are the main no-retraining answer to the map concern.


In [ ]:

def residual_summary(series: pd.Series) -> dict:
    r = pd.to_numeric(series, errors="coerce").dropna()
    if len(r) == 0:
        return {
            "residual_n": 0,
            "residual_bias": np.nan,
            "residual_mae": np.nan,
            "residual_rmse": np.nan,
            "residual_median_abs": np.nan,
            "residual_p90_abs": np.nan,
            "residual_p95_abs": np.nan,
            "residual_p99_abs": np.nan,
            "frac_abs_resid_gt_1c": np.nan,
            "frac_abs_resid_gt_2c": np.nan,
            "frac_abs_resid_gt_3c": np.nan,
        }
    a = r.abs()
    return {
        "residual_n": int(len(r)),
        "residual_bias": float(r.mean()),
        "residual_mae": float(a.mean()),
        "residual_rmse": float(np.sqrt(np.mean(np.square(r)))),
        "residual_median_abs": float(a.median()),
        "residual_p90_abs": float(a.quantile(0.90)),
        "residual_p95_abs": float(a.quantile(0.95)),
        "residual_p99_abs": float(a.quantile(0.99)),
        "frac_abs_resid_gt_1c": float((a > 1).mean()),
        "frac_abs_resid_gt_2c": float((a > 2).mean()),
        "frac_abs_resid_gt_3c": float((a > 3).mean()),
    }


def infer_prob_col(df: pd.DataFrame, manifest: Mapping | None = None, risk_model_key: str = "context_static_met") -> str | None:
    candidates = []
    if manifest is not None:
        if manifest.get("calibrated_prob_col"):
            candidates.append(manifest.get("calibrated_prob_col"))
        cfg = manifest.get("config", {})
        if cfg.get("prob_col"):
            candidates.append(cfg.get("prob_col"))
    candidates += [
        f"pred_ref_risk_prob_{risk_model_key}_reliability_calibrated",
        f"pred_ref_risk_prob_{risk_model_key}",
    ]
    candidates += [c for c in df.columns if c.startswith("pred_ref_risk_prob_") and "calibrated" in c]
    candidates += [c for c in df.columns if c.startswith("pred_ref_risk_prob_")]
    for c in candidates:
        if c in df.columns:
            return c
    return None


def infer_target_col(df: pd.DataFrame) -> str | None:
    for c in ["target_ref_risk_hiconf", "target_ref_risk"]:
        if c in df.columns:
            return c
    return None


def policy_specs(df: pd.DataFrame, manifest: Mapping | None = None, risk_model_key: str = "context_static_met") -> dict[str, pd.Series]:
    specs: dict[str, pd.Series] = {"raw_all_observed": pd.Series(True, index=df.index)}

    qc_cols = [c for c in df.columns if str(c).startswith("qc_") and str(c).endswith("_is_outlier")]
    for c in qc_cols:
        level = c.replace("qc_", "").replace("_is_outlier", "")
        q = pd.to_numeric(df[c], errors="coerce")
        specs[f"crowdqc_{level}_clean_available_only"] = q.notna() & q.eq(0)
        specs[f"crowdqc_{level}_clean_or_unavailable"] = q.fillna(0).eq(0)

    prob_col = infer_prob_col(df, manifest, risk_model_key=risk_model_key)
    if prob_col is not None:
        p = pd.to_numeric(df[prob_col], errors="coerce")
        for cutoff in [0.05, 0.15, 0.30, 0.50, 0.80]:
            specs[f"risk_{risk_model_key}_p_le_{cutoff:g}"] = p <= cutoff

    if "p_reject_as_unexplained_transient_error" in df.columns:
        p_reject = pd.to_numeric(df["p_reject_as_unexplained_transient_error"], errors="coerce")
        for cutoff in [0.15, 0.30, 0.50, 0.70]:
            specs[f"fusion_reject_score_p_le_{cutoff:g}"] = p_reject <= cutoff

    for name, col in [
        ("fusion_keep_conservative", "recommended_keep_conservative"),
        ("fusion_keep_microclimate", "recommended_keep_microclimate"),
        ("fusion_bias_correctable_only", "recommended_bias_correctable"),
    ]:
        if col in df.columns:
            specs[name] = as_bool(df[col])

    if "recommended_reject_transient_error" in df.columns:
        specs["fusion_remove_reject_transient_only"] = ~as_bool(df["recommended_reject_transient_error"])
    if "analysis_weight" in df.columns:
        specs["fusion_weight_positive"] = pd.to_numeric(df["analysis_weight"], errors="coerce").fillna(0) > 0

    return specs


def policy_metric_row(df: pd.DataFrame, keep: pd.Series, method: str, residual_col: str = "cws_ref_resid") -> dict:
    keep = pd.Series(keep, index=df.index).fillna(False).astype(bool)
    out = {
        "method": method,
        "n_total": int(len(df)),
        "n_kept": int(keep.sum()),
        "retention": float(keep.mean()) if len(keep) else np.nan,
        "n_stations_kept": int(df.loc[keep, ["network", "station_id"]].drop_duplicates().shape[0]) if {"network", "station_id"}.issubset(df.columns) else np.nan,
    }
    if residual_col in df.columns:
        out.update(residual_summary(df.loc[keep, residual_col]))

    target_col = infer_target_col(df)
    if target_col is not None:
        y = pd.to_numeric(df[target_col], errors="coerce")
        valid = y.isin([0, 1])
        if valid.any():
            yy = y[valid].astype(int)
            pred_bad = (~keep[valid]).astype(int)
            tp = int(((yy == 1) & (pred_bad == 1)).sum())
            fp = int(((yy == 0) & (pred_bad == 1)).sum())
            tn = int(((yy == 0) & (pred_bad == 0)).sum())
            fn = int(((yy == 1) & (pred_bad == 0)).sum())
            out.update({
                "label_n": int(valid.sum()),
                "target_event_rate": float(yy.mean()),
                "tn": tn, "fp": fp, "fn": fn, "tp": tp,
                "proxy_accuracy": float((tn + tp) / max(tn + fp + fn + tp, 1)),
                "proxy_precision_bad": float(tp / max(tp + fp, 1)),
                "proxy_recall_bad": float(tp / max(tp + fn, 1)),
                "proxy_f1_bad": float(2 * tp / max(2 * tp + fp + fn, 1)),
            })
    return out


def stratified_policy_metrics(df: pd.DataFrame, group_col: str, manifest: Mapping | None = None, risk_model_key: str = "context_static_met") -> pd.DataFrame:
    rows = []
    for group_value, g in df.groupby(group_col, dropna=False):
        specs = policy_specs(g, manifest=manifest, risk_model_key=risk_model_key)
        for method, keep in specs.items():
            row = policy_metric_row(g, keep, method=method, residual_col="cws_ref_resid")
            row = {group_col: str(group_value), **row}
            row["n_stations_total_in_group"] = int(g[["network", "station_id"]].drop_duplicates().shape[0])
            rows.append(row)
    return pd.DataFrame(rows)


def correction_specs(df: pd.DataFrame, manifest: Mapping | None = None) -> dict[tuple[str, str], pd.Series]:
    all_rows = pd.Series(True, index=df.index)
    specs: dict[tuple[str, str], pd.Series] = {
        ("raw_all_observed_no_bias_correction", "cws_ref_resid"): all_rows,
    }

    if "cws_ref_resid_corrected_station_bias_policy" in df.columns:
        specs[("all_rows_policy_bias_corrected", "cws_ref_resid_corrected_station_bias_policy")] = all_rows

    if "recommended_keep_microclimate" in df.columns:
        keep = as_bool(df["recommended_keep_microclimate"])
        specs[("fusion_keep_microclimate_no_bias_correction", "cws_ref_resid")] = keep
        if "cws_ref_resid_corrected_station_bias_policy" in df.columns:
            specs[("fusion_keep_microclimate_policy_bias_corrected", "cws_ref_resid_corrected_station_bias_policy")] = keep

    if "recommended_keep_conservative" in df.columns:
        keep = as_bool(df["recommended_keep_conservative"])
        specs[("fusion_keep_conservative_no_bias_correction", "cws_ref_resid")] = keep
        if "cws_ref_resid_corrected_station_bias_policy" in df.columns:
            specs[("fusion_keep_conservative_policy_bias_corrected", "cws_ref_resid_corrected_station_bias_policy")] = keep

    if "recommended_bias_correctable" in df.columns:
        keep = as_bool(df["recommended_bias_correctable"])
        specs[("fusion_bias_correctable_subset_no_bias_correction", "cws_ref_resid")] = keep
        if "cws_ref_resid_corrected_station_bias_policy" in df.columns:
            specs[("fusion_bias_correctable_subset_policy_bias_corrected", "cws_ref_resid_corrected_station_bias_policy")] = keep


    qc_cols = [c for c in df.columns if str(c).startswith("qc_") and str(c).endswith("_is_outlier")]
    for c in qc_cols:
        level = c.replace("qc_", "").replace("_is_outlier", "")
        q = pd.to_numeric(df[c], errors="coerce")
        for suffix, keep in {
            "clean_available_only": q.notna() & q.eq(0),
            "clean_or_unavailable": q.fillna(0).eq(0),
        }.items():
            specs[(f"crowdqc_{level}_{suffix}_no_bias_correction", "cws_ref_resid")] = keep
            if "cws_ref_resid_corrected_station_bias_policy" in df.columns:
                specs[(f"crowdqc_{level}_{suffix}_policy_bias_corrected", "cws_ref_resid_corrected_station_bias_policy")] = keep


    if "p_reject_as_unexplained_transient_error" in df.columns:
        p_reject = pd.to_numeric(
            df["p_reject_as_unexplained_transient_error"],
            errors="coerce",
        )

        for cutoff in [0.15, 0.30, 0.50, 0.70]:
            keep = p_reject <= cutoff

            specs[
                (f"fusion_reject_score_p_le_{cutoff:g}_no_bias_correction", "cws_ref_resid")
            ] = keep

            if "cws_ref_resid_corrected_station_bias_policy" in df.columns:
                specs[
                    (
                        f"fusion_reject_score_p_le_{cutoff:g}_policy_bias_corrected",
                        "cws_ref_resid_corrected_station_bias_policy",
                    )
                ] = keep
    return specs


def correction_metric_row(df: pd.DataFrame, keep: pd.Series, method: str, residual_col: str) -> dict:
    keep = pd.Series(keep, index=df.index).fillna(False).astype(bool)
    out = {
        "method": method,
        "residual_col": residual_col,
        "n_total": int(len(df)),
        "n_used": int(keep.sum()),
        "coverage": float(keep.mean()) if len(df) else np.nan,
        "n_stations_used": int(df.loc[keep, ["network", "station_id"]].drop_duplicates().shape[0]) if {"network", "station_id"}.issubset(df.columns) else np.nan,
    }
    if residual_col in df.columns:
        out.update(residual_summary(df.loc[keep, residual_col]))
    return out


def stratified_correction_metrics(df: pd.DataFrame, group_col: str, manifest: Mapping | None = None) -> pd.DataFrame:
    rows = []
    for group_value, g in df.groupby(group_col, dropna=False):
        specs = correction_specs(g, manifest=manifest)
        for (method, residual_col), keep in specs.items():
            row = correction_metric_row(g, keep, method=method, residual_col=residual_col)
            row = {group_col: str(group_value), **row}
            row["n_stations_total_in_group"] = int(g[["network", "station_id"]].drop_duplicates().shape[0])
            rows.append(row)
    return pd.DataFrame(rows)


policy_by_distance_domain = stratified_policy_metrics(eval_df, "ows_distance_domain", fusion_manifest, RISK_MODEL_KEY)
policy_by_support_class = stratified_policy_metrics(eval_df, "ref_support_class", fusion_manifest, RISK_MODEL_KEY)
correction_by_distance_domain = stratified_correction_metrics(eval_df, "ows_distance_domain", fusion_manifest)
correction_by_support_class = stratified_correction_metrics(eval_df, "ref_support_class", fusion_manifest)

save_df(policy_by_distance_domain, f"{CITY}_policy_metrics_by_ows_distance_domain.csv")
save_df(policy_by_support_class, f"{CITY}_policy_metrics_by_ref_support_class.csv")
save_df(correction_by_distance_domain, f"{CITY}_correction_metrics_by_ows_distance_domain.csv")
save_df(correction_by_support_class, f"{CITY}_correction_metrics_by_ref_support_class.csv")


main_methods = [
    "raw_all_observed",
    "crowdqc_strict_clean_available_only",
    f"risk_{RISK_MODEL_KEY}_p_le_0.15",
    "fusion_keep_conservative",
    "fusion_keep_microclimate",
    "fusion_remove_reject_transient_only",
]
display(policy_by_distance_domain[policy_by_distance_domain["method"].isin(main_methods)].sort_values(["ows_distance_domain", "method"]))


## 7. Fusion categories by support domain

This table shows whether weakly supported stations are being routed to `reference_limited_or_ows_support_review` or over-interpreted as bias/error/microclimate.


In [ ]:

def category_by_group(df: pd.DataFrame, group_col: str, category_col: str = "fusion_category") -> pd.DataFrame:
    if category_col not in df.columns:
        return pd.DataFrame()
    rows = []
    for group_value, g in df.groupby(group_col, dropna=False):
        total = len(g)
        for category, h in g.groupby(category_col, dropna=False):
            row = {
                group_col: str(group_value),
                category_col: str(category),
                "n": int(len(h)),
                "fraction_within_group": float(len(h) / total) if total else np.nan,
                "n_stations": int(h[["network", "station_id"]].drop_duplicates().shape[0]) if {"network", "station_id"}.issubset(h.columns) else np.nan,
            }
            if "cws_ref_resid" in h.columns:
                row.update(residual_summary(h["cws_ref_resid"]))
            rows.append(row)
    return pd.DataFrame(rows).sort_values([group_col, "fraction_within_group"], ascending=[True, False]).reset_index(drop=True)

category_by_distance_domain = category_by_group(eval_df, "ows_distance_domain")
category_by_support_class = category_by_group(eval_df, "ref_support_class")

save_df(category_by_distance_domain, f"{CITY}_fusion_category_by_ows_distance_domain.csv")
save_df(category_by_support_class, f"{CITY}_fusion_category_by_ref_support_class.csv")

display(category_by_distance_domain.head(40))


## 8. Identify categories that may be over-interpreted under weak support

This is the reviewer-facing diagnostic: among observations currently called correction/error/preserve candidates, how many are low/outside support?


In [ ]:

interpretation_sensitive_categories = {
    "systemic_station_bias_correction_candidate",
    "radiation_or_siting_bias_candidate",
    "environmental_difference_candidate_preserve",
    "possible_bias_or_microclimate_preserve_with_caution",
    "qc_confirmed_probable_transient_or_sensor_error",
    "qc_missed_probable_high_risk_observation",
}

sensitive = eval_df[eval_df["fusion_category"].astype(str).isin(interpretation_sensitive_categories)].copy()
rows = []
for cat, g in sensitive.groupby("fusion_category", dropna=False):
    rows.append({
        "fusion_category": str(cat),
        "n": int(len(g)),
        "fraction_of_eval_rows": float(len(g) / len(eval_df)),
        "frac_poor_support_strict": float(g["poor_ows_support_strict"].mean()),
        "frac_poor_support_sensitive": float(g["poor_ows_support_sensitive"].mean()),
        "median_nearest_ows_dist_km": float(pd.to_numeric(g["nearest_ows_dist_km"], errors="coerce").median()),
        "p90_nearest_ows_dist_km": float(pd.to_numeric(g["nearest_ows_dist_km"], errors="coerce").quantile(0.90)),
    })

sensitive_category_support_audit = pd.DataFrame(rows).sort_values("frac_poor_support_sensitive", ascending=False)
save_df(sensitive_category_support_audit, f"{CITY}_interpretation_sensitive_category_support_audit.csv")
display(sensitive_category_support_audit)


## 9. Conservative reference-support guardrail sensitivity

This does **not** change the ML model. It re-labels weakly supported, reference-disagreement cases as `reference_limited_or_ows_support_review` unless there is strong independent QC/transient evidence.

Use the strict guardrail for the main paper sensitivity; use the sensitive guardrail as an appendix stress test.


In [ ]:

def recompute_recommendation_flags(x: pd.DataFrame) -> pd.DataFrame:
    out = x.copy()
    cat = out["fusion_category"].astype(str)

    keep_conservative = {
        "high_confidence_reference_consistent",
        "likely_valid_low_unexplained_error",
    }
    keep_micro = {
        "high_confidence_reference_consistent",
        "likely_valid_low_unexplained_error",
        "qc_flagged_low_reference_risk_rescue_candidate",
        "systemic_station_bias_correction_candidate",
        "radiation_or_siting_bias_candidate",
        "possible_bias_or_microclimate_preserve_with_caution",
        "environmental_difference_candidate_preserve",
    }
    bias_correctable = {
        "systemic_station_bias_correction_candidate",
        "radiation_or_siting_bias_candidate",
    }
    reject = {
        "missing_or_unscored",
        "qc_confirmed_probable_transient_or_sensor_error",
        "qc_missed_probable_high_risk_observation",
    }

    action_map = {
        "missing_or_unscored": "exclude_missing",
        "high_confidence_reference_consistent": "use_as_is",
        "likely_valid_low_unexplained_error": "use_as_is_or_light_downweight",
        "qc_flagged_low_reference_risk_rescue_candidate": "rescue_use_with_caution",
        "systemic_station_bias_correction_candidate": "bias_correct_then_use",
        "radiation_or_siting_bias_candidate": "context_bias_correct_or_downweight",
        "possible_bias_or_microclimate_preserve_with_caution": "preserve_or_downweight",
        "environmental_difference_candidate_preserve": "preserve_for_microclimate_analysis",
        "reference_limited_or_ows_support_review": "review_reference_support_or_downweight",
        "qc_confirmed_probable_transient_or_sensor_error": "exclude_transient_error",
        "qc_missed_probable_high_risk_observation": "exclude_or_manual_review",
        "ambiguous_review": "manual_review_or_downweight",
    }

    out["recommended_action"] = cat.map(action_map).fillna("manual_review_or_downweight")
    out["recommended_keep_conservative"] = cat.isin(keep_conservative)
    out["recommended_keep_microclimate"] = cat.isin(keep_micro)
    out["recommended_bias_correctable"] = cat.isin(bias_correctable)
    out["recommended_reject_transient_error"] = cat.isin(reject)


    if {"temp_raw", "ref_mu", "expected_train_station_context_bias_c"}.issubset(out.columns):
        expected = pd.to_numeric(out["expected_train_station_context_bias_c"], errors="coerce")
        temp = pd.to_numeric(out["temp_raw"], errors="coerce")
        ref_mu = pd.to_numeric(out["ref_mu"], errors="coerce")
        applied = out["recommended_bias_correctable"].fillna(False) & expected.notna()
        out["station_bias_correction_applied"] = applied
        out["station_bias_correction_c"] = expected.where(applied, 0.0).astype("float32")
        out["temp_corrected_station_bias_policy"] = (temp - out["station_bias_correction_c"]).astype("float32")
        out["cws_ref_resid_corrected_station_bias_policy"] = (out["temp_corrected_station_bias_policy"] - ref_mu).astype("float32")


    if "p_reject_as_unexplained_transient_error" in out.columns:
        w = 1.0 - pd.to_numeric(out["p_reject_as_unexplained_transient_error"], errors="coerce").fillna(1.0)
    else:
        w = out["recommended_keep_microclimate"].astype(float)
    w = w.clip(0, 1)
    w.loc[out["recommended_keep_conservative"]] = 1.0
    w.loc[cat.eq("likely_valid_low_unexplained_error")] = np.maximum(w.loc[cat.eq("likely_valid_low_unexplained_error")], 0.85)
    w.loc[cat.eq("systemic_station_bias_correction_candidate")] = np.maximum(w.loc[cat.eq("systemic_station_bias_correction_candidate")], 0.70)
    w.loc[cat.eq("radiation_or_siting_bias_candidate")] = np.maximum(w.loc[cat.eq("radiation_or_siting_bias_candidate")], 0.65)
    w.loc[cat.eq("environmental_difference_candidate_preserve")] = np.maximum(w.loc[cat.eq("environmental_difference_candidate_preserve")], 0.65)
    w.loc[cat.eq("reference_limited_or_ows_support_review")] = np.minimum(w.loc[cat.eq("reference_limited_or_ows_support_review")], 0.50)
    w.loc[out["recommended_reject_transient_error"]] = 0.0
    out["analysis_weight"] = w.astype("float32")
    return out


def apply_reference_support_guardrail(df: pd.DataFrame, mode: str = "strict") -> tuple[pd.DataFrame, pd.DataFrame]:
    """Route weakly supported large-disagreement cases to reference-limited review.

    mode='strict': only >10 km / outside_support.
    mode='sensitive': >8 km / low_support / outside_support.
    """
    x = df.copy()
    if mode not in {"strict", "sensitive"}:
        raise ValueError("mode must be 'strict' or 'sensitive'")

    poor = x["poor_ows_support_strict"] if mode == "strict" else x["poor_ows_support_sensitive"]
    poor = poor.fillna(False).astype(bool)

    p_any = safe_num(x, "p_any_quality_issue", default=0).fillna(0)
    p_reject = safe_num(x, "p_reject_as_unexplained_transient_error", default=0).fillna(0)
    p_explained = safe_num(x, "p_residual_explained_by_persistent_station_context", default=0).fillna(0)
    p_env = safe_num(x, "p_environmental_difference_signal", default=0).fillna(0)
    p_rad = safe_num(x, "p_radiation_or_siting_bias_signal", default=0).fillna(0)
    abs_resid = safe_num(x, "abs_cws_ref_resid", default=np.nan)
    abs_z = safe_num(x, "abs_cws_ref_z", default=np.nan)

    qc_cols = [c for c in x.columns if str(c).startswith("qc_") and str(c).endswith("_is_outlier")]
    qc_lenient_or_strict = pd.Series(False, index=x.index)
    qc_any = pd.Series(False, index=x.index)
    for c in qc_cols:
        q = pd.to_numeric(x[c], errors="coerce").eq(1)
        qc_any |= q
        if "lenient" in c or ("strict" in c and "ultra" not in c):
            qc_lenient_or_strict |= q


    currently_clean_or_rescue = x["fusion_category"].astype(str).isin([
        "high_confidence_reference_consistent",
        "likely_valid_low_unexplained_error",
        "qc_flagged_low_reference_risk_rescue_candidate",
    ])

    has_reference_disagreement = (
        (p_any >= 0.30)
        | (abs_resid >= 2.0)
        | (abs_z >= 3.0)
        | x["fusion_category"].astype(str).isin(interpretation_sensitive_categories)
    )


    independent_error_evidence = (
        qc_lenient_or_strict
        & (p_reject >= 0.80)
        & (p_explained < 0.35)
        & (p_env < 0.35)
        & (p_rad < 0.40)
    )

    route_to_reference_limited = poor & has_reference_disagreement & ~currently_clean_or_rescue & ~independent_error_evidence

    before = x.loc[route_to_reference_limited, "fusion_category"].astype(str).value_counts(dropna=False)
    x.loc[route_to_reference_limited, "fusion_category"] = "reference_limited_or_ows_support_review"
    x.loc[route_to_reference_limited, "fusion_reason"] = (
        f"{mode} OWS-support guardrail: weak spatial reference support; avoid high-confidence error/bias/microclimate interpretation"
    )
    x = recompute_recommendation_flags(x)

    audit = pd.DataFrame({
        "guardrail_mode": mode,
        "original_category": before.index.astype(str),
        "n_rerouted_to_reference_limited": before.values,
    })
    audit["total_rerouted"] = int(route_to_reference_limited.sum())
    audit["fraction_eval_rows_rerouted"] = float(route_to_reference_limited.mean()) if len(route_to_reference_limited) else np.nan
    return x, audit


guard_strict_df, guard_strict_audit = apply_reference_support_guardrail(eval_df, mode="strict")
guard_sensitive_df, guard_sensitive_audit = apply_reference_support_guardrail(eval_df, mode="sensitive")

save_df(guard_strict_audit, f"{CITY}_support_guardrail_strict_reroute_audit.csv")
save_df(guard_sensitive_audit, f"{CITY}_support_guardrail_sensitive_reroute_audit.csv")

display(guard_strict_audit)
display(guard_sensitive_audit)


## 10. Compare original vs guardrailed fusion


In [ ]:

def tag_version(df: pd.DataFrame, version: str) -> pd.DataFrame:
    out = df.copy()
    out.insert(0, "fusion_version", version)
    return out

original_policy = tag_version(stratified_policy_metrics(eval_df, "ows_distance_domain", fusion_manifest, RISK_MODEL_KEY), "original")
strict_policy = tag_version(stratified_policy_metrics(guard_strict_df, "ows_distance_domain", fusion_manifest, RISK_MODEL_KEY), "support_guardrail_strict")
sensitive_policy = tag_version(stratified_policy_metrics(guard_sensitive_df, "ows_distance_domain", fusion_manifest, RISK_MODEL_KEY), "support_guardrail_sensitive")
policy_guardrail_comparison = pd.concat([original_policy, strict_policy, sensitive_policy], ignore_index=True)

original_correction = tag_version(stratified_correction_metrics(eval_df, "ows_distance_domain", fusion_manifest), "original")
strict_correction = tag_version(stratified_correction_metrics(guard_strict_df, "ows_distance_domain", fusion_manifest), "support_guardrail_strict")
sensitive_correction = tag_version(stratified_correction_metrics(guard_sensitive_df, "ows_distance_domain", fusion_manifest), "support_guardrail_sensitive")
correction_guardrail_comparison = pd.concat([original_correction, strict_correction, sensitive_correction], ignore_index=True)

original_category = tag_version(category_by_group(eval_df, "ows_distance_domain"), "original")
strict_category = tag_version(category_by_group(guard_strict_df, "ows_distance_domain"), "support_guardrail_strict")
sensitive_category = tag_version(category_by_group(guard_sensitive_df, "ows_distance_domain"), "support_guardrail_sensitive")
category_guardrail_comparison = pd.concat([original_category, strict_category, sensitive_category], ignore_index=True)

save_df(policy_guardrail_comparison, f"{CITY}_policy_metrics_by_distance_domain__original_vs_guardrails.csv")
save_df(correction_guardrail_comparison, f"{CITY}_correction_metrics_by_distance_domain__original_vs_guardrails.csv")
save_df(category_guardrail_comparison, f"{CITY}_category_by_distance_domain__original_vs_guardrails.csv")


display(
    policy_guardrail_comparison[
        policy_guardrail_comparison["method"].isin([
            "raw_all_observed",
            "fusion_reject_score_p_le_0.15",
            "fusion_reject_score_p_le_0.3",
            "fusion_reject_score_p_le_0.5",
            "fusion_keep_conservative",
            "fusion_keep_microclimate",
            "fusion_remove_reject_transient_only",
        ])
    ].sort_values(["ows_distance_domain", "method", "fusion_version"])
)

display(
    correction_guardrail_comparison[
        correction_guardrail_comparison["method"].isin([
            "raw_all_observed_no_bias_correction",
            "crowdqc_lenient_clean_available_only_policy_bias_corrected",
            "crowdqc_strict_clean_available_only_policy_bias_corrected",
            "crowdqc_ultra_strict_clean_available_only_policy_bias_corrected",
            "fusion_reject_score_p_le_0.15_policy_bias_corrected",
            "fusion_reject_score_p_le_0.3_policy_bias_corrected",
            "fusion_reject_score_p_le_0.5_policy_bias_corrected",
            "fusion_keep_microclimate_policy_bias_corrected",
        ])
    ].sort_values(["ows_distance_domain", "method", "fusion_version"])
)


## 11. Station-level dominance by spatial support

This checks whether a few weakly supported stations dominate apparent gains or failures.


In [ ]:

def station_support_diagnostics(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["station_key"] = station_key_df(x)
    x["abs_resid"] = pd.to_numeric(x["cws_ref_resid"], errors="coerce").abs()
    if "cws_ref_resid_corrected_station_bias_policy" in x.columns:
        x["abs_resid_corrected_policy"] = pd.to_numeric(x["cws_ref_resid_corrected_station_bias_policy"], errors="coerce").abs()
    else:
        x["abs_resid_corrected_policy"] = np.nan

    bool_cols = [
        "recommended_keep_conservative",
        "recommended_keep_microclimate",
        "recommended_bias_correctable",
        "recommended_reject_transient_error",
        "poor_ows_support_strict",
        "poor_ows_support_sensitive",
    ]
    for c in bool_cols:
        if c in x.columns:
            x[c + "__bool"] = as_bool(x[c]) if c.startswith("recommended") else x[c].fillna(False).astype(bool)

    agg = {
        "n_rows": ("station_key", "size"),
        "network": ("network", "first"),
        "station_id": ("station_id", "first"),
        "nearest_ows_dist_km": ("nearest_ows_dist_km", "median"),
        "ows_distance_domain": ("ows_distance_domain", lambda s: s.astype(str).mode().iloc[0] if len(s.mode()) else str(s.iloc[0])),
        "ref_support_class": ("ref_support_class", lambda s: s.astype(str).mode().iloc[0] if len(s.mode()) else str(s.iloc[0])),
        "raw_residual_bias": ("cws_ref_resid", lambda s: pd.to_numeric(s, errors="coerce").mean()),
        "raw_residual_mae": ("abs_resid", "mean"),
        "raw_residual_p95_abs": ("abs_resid", lambda s: pd.to_numeric(s, errors="coerce").quantile(0.95)),
        "policy_corrected_mae": ("abs_resid_corrected_policy", "mean"),
    }
    for c in bool_cols:
        bc = c + "__bool"
        if bc in x.columns:
            agg[f"frac_{c}"] = (bc, "mean")

    station = x.groupby("station_key", dropna=False).agg(**agg).reset_index()
    station["policy_correction_mae_gain"] = station["raw_residual_mae"] - station["policy_corrected_mae"]
    station["raw_abs_error_contribution"] = station["raw_residual_mae"] * station["n_rows"]
    total_error = station["raw_abs_error_contribution"].sum()
    station["share_total_abs_error"] = station["raw_abs_error_contribution"] / total_error if total_error else np.nan
    return station.sort_values("raw_abs_error_contribution", ascending=False).reset_index(drop=True)

station_diag = station_support_diagnostics(eval_df)
save_df(station_diag, f"{CITY}_station_level_spatial_support_diagnostics.csv")


rows = []
for domain, g in station_diag.groupby("ows_distance_domain", dropna=False):
    total_rows = g["n_rows"].sum()
    total_error = g["raw_abs_error_contribution"].sum()
    row = {
        "ows_distance_domain": str(domain),
        "n_stations": int(len(g)),
        "n_rows": int(total_rows),
        "mean_station_raw_mae": float(g["raw_residual_mae"].mean()),
        "row_weighted_raw_mae": float(total_error / total_rows) if total_rows else np.nan,
        "top10_station_row_share": float(g.head(10)["n_rows"].sum() / total_rows) if total_rows else np.nan,
        "top10_station_abs_error_share": float(g.head(10)["raw_abs_error_contribution"].sum() / total_error) if total_error else np.nan,
    }
    rows.append(row)
station_dominance_by_domain = pd.DataFrame(rows)
save_df(station_dominance_by_domain, f"{CITY}_station_dominance_by_ows_distance_domain.csv")
display(station_dominance_by_domain)


## 12. Optional station-bootstrap CIs by support domain

Set `RUN_BOOTSTRAP = True` if you want CIs. This can take a few minutes on the full test table.


In [ ]:

RUN_BOOTSTRAP = True
N_BOOTSTRAP = 300
BOOTSTRAP_METHODS = [
    "raw_all_observed",
    "crowdqc_strict_clean_available_only",
    f"risk_{RISK_MODEL_KEY}_p_le_0.15",
    "fusion_keep_conservative",
    "fusion_keep_microclimate",
    "fusion_remove_reject_transient_only",
]


def bootstrap_policy_by_station(df: pd.DataFrame, group_col: str, n_bootstrap: int = 300, random_state: int = 2026) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    rows = []
    for group_value, g in df.groupby(group_col, dropna=False):
        specs = policy_specs(g, fusion_manifest, RISK_MODEL_KEY)
        station_key = station_key_df(g)
        stations = pd.Index(station_key.unique())
        if len(stations) == 0:
            continue

        for method in BOOTSTRAP_METHODS:
            if method not in specs:
                continue
            keep = pd.Series(specs[method], index=g.index).fillna(False).astype(bool)
            resid = pd.to_numeric(g["cws_ref_resid"], errors="coerce")
            tmp = pd.DataFrame({
                "station_key": station_key.values,
                "n_total": 1,
                "n_kept": keep.astype(int).values,
                "sum_abs_resid_kept": resid.abs().where(keep, 0).fillna(0).values,
                "sum_sq_resid_kept": (resid ** 2).where(keep, 0).fillna(0).values,
            })
            block = tmp.groupby("station_key", sort=False).sum(numeric_only=True).reindex(stations).fillna(0)
            arr = block[["n_total", "n_kept", "sum_abs_resid_kept", "sum_sq_resid_kept"]].to_numpy(dtype=float)
            idx = rng.integers(0, len(stations), size=(n_bootstrap, len(stations)))
            draws = arr[idx].sum(axis=1)
            n_total = draws[:, 0]
            n_kept = draws[:, 1]
            retention = np.divide(n_kept, n_total, out=np.full_like(n_kept, np.nan), where=n_total > 0)
            mae = np.divide(draws[:, 2], n_kept, out=np.full_like(n_kept, np.nan), where=n_kept > 0)
            rmse = np.sqrt(np.divide(draws[:, 3], n_kept, out=np.full_like(n_kept, np.nan), where=n_kept > 0))
            rows.append({
                group_col: str(group_value),
                "method": method,
                "n_stations_blocks": int(len(stations)),
                "n_bootstrap": int(n_bootstrap),
                "retention_ci_low": float(np.nanquantile(retention, 0.025)),
                "retention_boot_median": float(np.nanquantile(retention, 0.50)),
                "retention_ci_high": float(np.nanquantile(retention, 0.975)),
                "mae_ci_low": float(np.nanquantile(mae, 0.025)),
                "mae_boot_median": float(np.nanquantile(mae, 0.50)),
                "mae_ci_high": float(np.nanquantile(mae, 0.975)),
                "rmse_ci_low": float(np.nanquantile(rmse, 0.025)),
                "rmse_boot_median": float(np.nanquantile(rmse, 0.50)),
                "rmse_ci_high": float(np.nanquantile(rmse, 0.975)),
            })
    return pd.DataFrame(rows)

if RUN_BOOTSTRAP:
    boot_by_domain = bootstrap_policy_by_station(eval_df, "ows_distance_domain", n_bootstrap=N_BOOTSTRAP)
    save_df(boot_by_domain, f"{CITY}_policy_station_bootstrap_ci_by_ows_distance_domain.csv")
    display(boot_by_domain)
else:
    print("Bootstrap skipped. Set RUN_BOOTSTRAP = True to run it.")


## 13. Simple paper plots


In [ ]:


plot_df = support_by_domain.copy()
fig = plt.figure(figsize=(8, 4))
plt.bar(plot_df["ows_distance_domain"], plot_df["row_fraction"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Fraction of test observations")
plt.title("CWS test observations by OWS-support distance domain")
plt.tight_layout()
fig_path = OUTPUT_DIR / f"{CITY}_test_observations_by_ows_support_domain.png"
plt.savefig(fig_path, dpi=200)
plt.show()
print("saved:", fig_path)


plot_methods = ["raw_all_observed", "crowdqc_strict_clean_available_only", "fusion_keep_conservative", "fusion_keep_microclimate"]
plot = policy_by_distance_domain[policy_by_distance_domain["method"].isin(plot_methods)].copy()
pivot = plot.pivot(index="ows_distance_domain", columns="method", values="residual_mae")
fig = plt.figure(figsize=(9, 4))
pivot.plot(kind="bar", ax=plt.gca())
plt.ylabel("MAE vs OWS reference (°C)")
plt.title("Residual error by OWS-support domain")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
fig_path = OUTPUT_DIR / f"{CITY}_mae_by_ows_support_domain_key_methods.png"
plt.savefig(fig_path, dpi=200)
plt.show()
print("saved:", fig_path)


## 14. Export a compact paper-ready summary


In [ ]:


paper_methods = [
    "raw_all_observed",
    "crowdqc_lenient_clean_available_only",
    "crowdqc_strict_clean_available_only",
    "crowdqc_ultra_strict_clean_available_only",
    f"risk_{RISK_MODEL_KEY}_p_le_0.15",
    "fusion_keep_conservative",
    "fusion_keep_microclimate",
    "fusion_remove_reject_transient_only",
]

paper_cols = [
    "ows_distance_domain", "method", "n_total", "n_kept", "retention", "n_stations_kept",
    "residual_mae", "residual_rmse", "residual_p95_abs", "frac_abs_resid_gt_3c",
    "target_event_rate", "proxy_f1_bad",
]

paper_spatial_summary = policy_by_distance_domain[
    policy_by_distance_domain["method"].isin(paper_methods)
].copy()
paper_spatial_summary = paper_spatial_summary[[c for c in paper_cols if c in paper_spatial_summary.columns]]

paper_correction_methods = [
    "raw_all_observed_no_bias_correction",

    "crowdqc_lenient_clean_available_only_policy_bias_corrected",
    "crowdqc_strict_clean_available_only_policy_bias_corrected",
    "crowdqc_ultra_strict_clean_available_only_policy_bias_corrected",

    "fusion_reject_score_p_le_0.15_policy_bias_corrected",
    "fusion_reject_score_p_le_0.3_policy_bias_corrected",
    "fusion_reject_score_p_le_0.5_policy_bias_corrected",

    "fusion_keep_microclimate_policy_bias_corrected",
]

paper_correction_cols = [
    "ows_distance_domain",
    "method",
    "residual_col",
    "n_total",
    "n_used",
    "coverage",
    "n_stations_used",
    "residual_mae",
    "residual_rmse",
    "residual_p95_abs",
    "frac_abs_resid_gt_3c",
]

paper_spatial_correction_summary = correction_by_distance_domain[
    correction_by_distance_domain["method"].isin(paper_correction_methods)
].copy()

paper_spatial_correction_summary = paper_spatial_correction_summary[
    [c for c in paper_correction_cols if c in paper_spatial_correction_summary.columns]
]

save_df(
    paper_spatial_correction_summary,
    f"{CITY}_PAPER_spatial_support_corrected_frontier_summary.csv",
)

display(paper_spatial_correction_summary)

save_df(paper_spatial_summary, f"{CITY}_PAPER_spatial_support_policy_summary.csv")

guardrail_paper_summary = policy_guardrail_comparison[
    policy_guardrail_comparison["method"].isin(["fusion_keep_conservative", "fusion_keep_microclimate", "fusion_remove_reject_transient_only"])
].copy()
save_df(guardrail_paper_summary, f"{CITY}_PAPER_guardrail_sensitivity_summary.csv")

display(paper_spatial_summary)
display(guardrail_paper_summary.head(30))

print("Done. Outputs written to:", OUTPUT_DIR)
